# Direct API Gateway → Lambda → Bedrock Tool-Calling Test

This notebook deliberately bypasses:

- LangChain
- LangGraph
- ChatOllama
- OpenAI protocol
- the EC2/Ollama wrapper

The purpose is to verify that **API Gateway → Lambda → Bedrock Converse** supports native Bedrock tool calling correctly.

## What we will prove

**Step 1:** Send a Bedrock-native request containing `toolConfig`.

Expected result:
- `stopReason == "tool_use"`
- `output.message.content[]` contains a structured `toolUse`

**Step 2:** Execute the requested Python tool locally.

**Step 3:** Send the tool result back using Bedrock-native `toolResult`.

Expected result:
- Bedrock returns the final answer
- the final answer uses the tool-derived value, not a hallucinated estimate


## 1. Install/import dependencies

If needed, run:

```python
%pip install requests python-dotenv
```


In [ ]:
# Uncomment if needed
# %pip install requests python-dotenv


In [19]:
import json
import os
from typing import Any, Dict

import requests
from dotenv import load_dotenv

load_dotenv()


False

## 2. Configure the API Gateway endpoint

Put these values in your `.env` file:

```text
LLM_GATEWAY_URL=https://<api-id>.execute-api.ap-southeast-1.amazonaws.com/UAT/<resource>
LLM_GATEWAY_API_KEY=<your-team-api-key>
LLM_GATEWAY_MODEL=global.anthropic.claude-sonnet-4-5-20250929-v1:0
```

This notebook does **not** hard-code your API key.


In [ ]:
API_URL = os.getenv("LLM_GATEWAY_URL")
API_KEY = os.getenv("LLM_GATEWAY_API_KEY")

MODEL = os.getenv(
    "LLM_GATEWAY_MODEL",
    "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
)

if not all([API_URL, API_KEY]):
    raise EnvironmentError(
        "Missing required env vars. Please create a .env file with:\n"
        "  LLM_GATEWAY_URL, LLM_GATEWAY_API_KEY\n"
        "See README.md for details."
    )

HEADERS = {
    "Content-Type": "application/json",
    "X-API-Key": API_KEY,
}

print("API URL:", API_URL)
print("Model:", MODEL)
print("API key loaded:", bool(API_KEY))

## 3. Define the system prompt and user request


In [21]:
SYSTEM_TEXT = '''You are a travel cost estimation agent.

Rules:
- When the user asks for total trip cost, ALWAYS call estimate_trip_cost.
- Do not invent cost figures.
- Use the tool result as the source of truth.

Output format:
1) Total cost (with assumptions)
'''

USER_TEXT = (
    "Plan a 2-day Tokyo trip for 2 adults. "
    "Mid comfort. How much will the trip cost?"
)

print(USER_TEXT)


Plan a 2-day Tokyo trip for 2 adults. Mid comfort. How much will the trip cost?


## 4. Define the local Python tool

This tool runs locally in the notebook. Bedrock should only decide **when/how to call it**.


In [22]:
def estimate_trip_cost(
    destination: str,
    days: int,
    travelers: int,
    comfort: str = "mid",
) -> Dict[str, Any]:

    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()

    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    lodging_pppd = {"budget": 60, "mid": 140, "premium": 300}[comfort]
    food_pppd = {"budget": 30, "mid": 60, "premium": 120}[comfort]
    local_transport_pppd = {"budget": 10, "mid": 20, "premium": 50}[comfort]
    activities_pppd = {"budget": 20, "mid": 50, "premium": 120}[comfort]

    lodging = lodging_pppd * travelers * days
    food = food_pppd * travelers * days
    transport = local_transport_pppd * travelers * days
    activities = activities_pppd * travelers * days

    subtotal = lodging + food + transport + activities
    contingency = round(subtotal * 0.12)
    total = subtotal + contingency

    return {
        "destination": destination,
        "days": days,
        "travelers": travelers,
        "comfort": comfort,
        "currency": "SGD",
        "breakdown": {
            "lodging": lodging,
            "food": food,
            "local_transport": transport,
            "activities": activities,
            "contingency": contingency,
        },
        "total_estimate": total,
        "note": "Heuristic estimate excludes international flights/insurance/visa fees.",
    }


### Sanity-check the tool directly

For 2 adults × 2 days × mid comfort, the expected result is **SGD 1,210**.


In [23]:
expected = estimate_trip_cost(
    destination="Tokyo",
    days=2,
    travelers=2,
    comfort="mid",
)

print(json.dumps(expected, indent=2))


{
  "destination": "Tokyo",
  "days": 2,
  "travelers": 2,
  "comfort": "mid",
  "currency": "SGD",
  "breakdown": {
    "lodging": 560,
    "food": 240,
    "local_transport": 80,
    "activities": 200,
    "contingency": 130
  },
  "total_estimate": 1210,
  "note": "Heuristic estimate excludes international flights/insurance/visa fees."
}


## 5. Define the Bedrock-native tool schema

This is the format your **Lambda expects** in `toolConfig`.


In [24]:
TOOL_CONFIG = {
    "tools": [
        {
            "toolSpec": {
                "name": "estimate_trip_cost",
                "description": (
                    "Estimate a rough trip budget in SGD for a destination, "
                    "number of days, number of travelers, and comfort level."
                ),
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "destination": {
                                "type": "string",
                                "description": "Trip destination",
                            },
                            "days": {
                                "type": "integer",
                                "description": "Number of trip days",
                            },
                            "travelers": {
                                "type": "integer",
                                "description": "Number of travelers",
                            },
                            "comfort": {
                                "type": "string",
                                "enum": ["budget", "mid", "premium"],
                                "description": "Travel comfort level",
                            },
                        },
                        "required": [
                            "destination",
                            "days",
                            "travelers",
                            "comfort",
                        ],
                    }
                },
            }
        }
    ]
}

print(json.dumps(TOOL_CONFIG, indent=2))


{
  "tools": [
    {
      "toolSpec": {
        "name": "estimate_trip_cost",
        "description": "Estimate a rough trip budget in SGD for a destination, number of days, number of travelers, and comfort level.",
        "inputSchema": {
          "json": {
            "type": "object",
            "properties": {
              "destination": {
                "type": "string",
                "description": "Trip destination"
              },
              "days": {
                "type": "integer",
                "description": "Number of trip days"
              },
              "travelers": {
                "type": "integer",
                "description": "Number of travelers"
              },
              "comfort": {
                "type": "string",
                "enum": [
                  "budget",
                  "mid",
                  "premium"
                ],
                "description": "Travel comfort level"
              }
            },
            "r

## 6. Helper to call API Gateway

This posts JSON directly to API Gateway and prints the raw response.


In [25]:
def post_gateway(payload: dict) -> dict:
    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload,
        timeout=180,
    )

    print("HTTP status:", response.status_code)

    try:
        body = response.json()
    except ValueError:
        print(response.text)
        response.raise_for_status()
        raise RuntimeError("Gateway did not return JSON")

    print(json.dumps(body, indent=2, ensure_ascii=False))

    if not response.ok:
        raise RuntimeError(
            f"Gateway request failed: HTTP {response.status_code}"
        )

    return body


# STEP 1 — Ask Bedrock to call the tool

Important message rules:

- `system` is top-level in Bedrock Converse
- `messages[].role` is `user` or `assistant`
- text is inside `content: [{"text": "..."}]`
- tools are in top-level `toolConfig`


In [26]:
initial_user_message = {
    "role": "user",
    "content": [
        {
            "text": USER_TEXT
        }
    ],
}

step1_request = {
    "modelId": MODEL,

    "system": [
        {
            "text": SYSTEM_TEXT
        }
    ],

    "messages": [
        initial_user_message
    ],

    "toolConfig": TOOL_CONFIG,

    "inferenceConfig": {
        "temperature": 0.0,
        "maxTokens": 500,
    },
}

print(json.dumps(step1_request, indent=2))


{
  "modelId": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
  "system": [
    {
      "text": "You are a travel cost estimation agent.\n\nRules:\n- When the user asks for total trip cost, ALWAYS call estimate_trip_cost.\n- Do not invent cost figures.\n- Use the tool result as the source of truth.\n\nOutput format:\n1) Total cost (with assumptions)\n"
    }
  ],
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "text": "Plan a 2-day Tokyo trip for 2 adults. Mid comfort. How much will the trip cost?"
        }
      ]
    }
  ],
  "toolConfig": {
    "tools": [
      {
        "toolSpec": {
          "name": "estimate_trip_cost",
          "description": "Estimate a rough trip budget in SGD for a destination, number of days, number of travelers, and comfort level.",
          "inputSchema": {
            "json": {
              "type": "object",
              "properties": {
                "destination": {
                  "type": "string",
     

## 7. POST Step 1 request to API Gateway


In [27]:
step1_response = post_gateway(step1_request)


HTTP status: 200
{
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "toolUse": {
            "toolUseId": "tooluse_qDSJ661wCxW0Ce1dcMk8c4",
            "name": "estimate_trip_cost",
            "input": {
              "destination": "Tokyo",
              "days": 2,
              "travelers": 2,
              "comfort": "mid"
            },
            "type": "tool_use"
          }
        }
      ]
    }
  },
  "stopReason": "tool_use",
  "usage": {
    "inputTokens": 750,
    "outputTokens": 106,
    "totalTokens": 856,
    "cacheReadInputTokens": 0,
    "cacheWriteInputTokens": 0
  },
  "metrics": {
    "latencyMs": 2132
  },
  "gateway": {
    "requestId": "13c8d419-a1e2-4e71-ba12-db4b719f91e1",
    "teamId": "TEAM-001",
    "reservedByThisRequest": 500,
    "chargedTokens": 856,
    "quotaSettlement": "SETTLED",
    "tokenLimit": 250000,
    "usedTokens": 75889,
    "reservedTokens": 0,
    "availableTokens": 174111
  }
}


## 8. Inspect whether Bedrock returned a structured `toolUse`

A correct response should resemble:

```json
{
  "stopReason": "tool_use",
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "toolUse": {
            "toolUseId": "...",
            "name": "estimate_trip_cost",
            "input": {
              "destination": "Tokyo",
              "days": 2,
              "travelers": 2,
              "comfort": "mid"
            }
          }
        }
      ]
    }
  }
}
```


In [28]:
stop_reason = step1_response.get("stopReason")

assistant_tool_message = (
    step1_response
    .get("output", {})
    .get("message")
)

tool_uses = []

if assistant_tool_message:
    for block in assistant_tool_message.get("content", []):
        if "toolUse" in block:
            tool_uses.append(block["toolUse"])

print("stopReason:", stop_reason)
print("toolUse count:", len(tool_uses))
print(json.dumps(assistant_tool_message, indent=2))

if not tool_uses:
    raise RuntimeError(
        "STEP 1 FAILED: no structured Bedrock toolUse returned."
    )

print("STEP 1 PASSED")


stopReason: tool_use
toolUse count: 1
{
  "role": "assistant",
  "content": [
    {
      "toolUse": {
        "toolUseId": "tooluse_qDSJ661wCxW0Ce1dcMk8c4",
        "name": "estimate_trip_cost",
        "input": {
          "destination": "Tokyo",
          "days": 2,
          "travelers": 2,
          "comfort": "mid"
        },
        "type": "tool_use"
      }
    }
  ]
}
STEP 1 PASSED


## 9. Extract the requested tool call

Preserve the `toolUseId`. The same ID must be used when sending the tool result back.


In [29]:
tool_use = tool_uses[0]

tool_use_id = tool_use["toolUseId"]
tool_name = tool_use["name"]
tool_args = tool_use["input"]

print("toolUseId:", tool_use_id)
print("tool name:", tool_name)
print("arguments:")
print(json.dumps(tool_args, indent=2))


toolUseId: tooluse_qDSJ661wCxW0Ce1dcMk8c4
tool name: estimate_trip_cost
arguments:
{
  "destination": "Tokyo",
  "days": 2,
  "travelers": 2,
  "comfort": "mid"
}


# STEP 2 — Execute the requested tool locally


In [30]:
if tool_name != "estimate_trip_cost":
    raise RuntimeError(f"Unexpected tool requested: {tool_name}")

tool_result = estimate_trip_cost(**tool_args)

print(json.dumps(tool_result, indent=2))


{
  "destination": "Tokyo",
  "days": 2,
  "travelers": 2,
  "comfort": "mid",
  "currency": "SGD",
  "breakdown": {
    "lodging": 560,
    "food": 240,
    "local_transport": 80,
    "activities": 200,
    "contingency": 130
  },
  "total_estimate": 1210,
  "note": "Heuristic estimate excludes international flights/insurance/visa fees."
}


# STEP 3 — Send the tool result back to Bedrock

This is the most important Bedrock message format.

Bedrock does **not** use:

```json
{"role": "tool"}
```

Instead, the result is sent as a **user message** containing a `toolResult`.

Also preserve the preceding assistant message containing the original `toolUse`.


In [31]:
tool_result_message = {
    "role": "user",
    "content": [
        {
            "toolResult": {
                "toolUseId": tool_use_id,
                "content": [
                    {
                        "json": tool_result
                    }
                ],
                "status": "success",
            }
        }
    ],
}

print(json.dumps(tool_result_message, indent=2))


{
  "role": "user",
  "content": [
    {
      "toolResult": {
        "toolUseId": "tooluse_qDSJ661wCxW0Ce1dcMk8c4",
        "content": [
          {
            "json": {
              "destination": "Tokyo",
              "days": 2,
              "travelers": 2,
              "comfort": "mid",
              "currency": "SGD",
              "breakdown": {
                "lodging": 560,
                "food": 240,
                "local_transport": 80,
                "activities": 200,
                "contingency": 130
              },
              "total_estimate": 1210,
              "note": "Heuristic estimate excludes international flights/insurance/visa fees."
            }
          }
        ],
        "status": "success"
      }
    }
  ]
}


## 10. Build the full second Converse request

The second request contains:

1. original user message
2. assistant message containing `toolUse`
3. user message containing `toolResult`


In [32]:
step2_request = {
    "modelId": MODEL,

    "system": [
        {
            "text": SYSTEM_TEXT
        }
    ],

    "messages": [
        initial_user_message,

        # Preserve EXACT Bedrock assistant message from Step 1
        assistant_tool_message,

        # Native Bedrock toolResult
        tool_result_message,
    ],

    "toolConfig": TOOL_CONFIG,

    "inferenceConfig": {
        "temperature": 0.0,
        "maxTokens": 500,
    },
}

print(json.dumps(step2_request, indent=2))


{
  "modelId": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
  "system": [
    {
      "text": "You are a travel cost estimation agent.\n\nRules:\n- When the user asks for total trip cost, ALWAYS call estimate_trip_cost.\n- Do not invent cost figures.\n- Use the tool result as the source of truth.\n\nOutput format:\n1) Total cost (with assumptions)\n"
    }
  ],
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "text": "Plan a 2-day Tokyo trip for 2 adults. Mid comfort. How much will the trip cost?"
        }
      ]
    },
    {
      "role": "assistant",
      "content": [
        {
          "toolUse": {
            "toolUseId": "tooluse_qDSJ661wCxW0Ce1dcMk8c4",
            "name": "estimate_trip_cost",
            "input": {
              "destination": "Tokyo",
              "days": 2,
              "travelers": 2,
              "comfort": "mid"
            },
            "type": "tool_use"
          }
        }
      ]
    },
    {
      "ro

## 11. POST Step 2 request to API Gateway


In [33]:
step2_response = post_gateway(step2_request)


HTTP status: 200
{
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "## Total Trip Cost: **SGD 1,210**\n\n### Cost Breakdown (for 2 adults, 2 days, mid comfort):\n- **Lodging**: SGD 560\n- **Food**: SGD 240\n- **Activities**: SGD 200\n- **Local Transport**: SGD 80\n- **Contingency**: SGD 130\n\n### Assumptions:\n- Mid-range comfort level (3-star hotels, mix of local restaurants and mid-range dining)\n- Estimate **excludes** international flights, travel insurance, and visa fees\n- Includes local transportation within Tokyo (trains, subways, taxis)\n- Covers typical tourist activities and sightseeing\n\nThis is a heuristic estimate to help you plan your budget. Actual costs may vary based on specific choices for accommodation, dining preferences, and activities."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 943,
    "outputTokens": 193,
    "totalTokens": 1136,
    "cacheReadInputTokens":

## 12. Inspect the final assistant answer


In [34]:
final_message = (
    step2_response
    .get("output", {})
    .get("message", {})
)

print(json.dumps(final_message, indent=2))

final_text = "\n".join(
    block["text"]
    for block in final_message.get("content", [])
    if "text" in block
)

print("\nFINAL ANSWER:")
print(final_text)


{
  "role": "assistant",
  "content": [
    {
      "text": "## Total Trip Cost: **SGD 1,210**\n\n### Cost Breakdown (for 2 adults, 2 days, mid comfort):\n- **Lodging**: SGD 560\n- **Food**: SGD 240\n- **Activities**: SGD 200\n- **Local Transport**: SGD 80\n- **Contingency**: SGD 130\n\n### Assumptions:\n- Mid-range comfort level (3-star hotels, mix of local restaurants and mid-range dining)\n- Estimate **excludes** international flights, travel insurance, and visa fees\n- Includes local transportation within Tokyo (trains, subways, taxis)\n- Covers typical tourist activities and sightseeing\n\nThis is a heuristic estimate to help you plan your budget. Actual costs may vary based on specific choices for accommodation, dining preferences, and activities."
    }
  ]
}

FINAL ANSWER:
## Total Trip Cost: **SGD 1,210**

### Cost Breakdown (for 2 adults, 2 days, mid comfort):
- **Lodging**: SGD 560
- **Food**: SGD 240
- **Activities**: SGD 200
- **Local Transport**: SGD 80
- **Contingency**:

## 13. Final validation

The tool result for this scenario is **SGD 1,210**.

If this notebook returns the correct structured tool call and final answer, then:

```text
API Gateway → Lambda → Bedrock tool calling = WORKING
```



In [17]:
if "1210" in final_text or "1,210" in final_text:
    print("PASS: final answer contains the tool-derived value SGD 1,210.")
else:
    print(
        "CHECK: protocol completed, but final text does not obviously "
        "contain SGD 1,210."
    )


PASS: final answer contains the tool-derived value SGD 1,210.
